In [1]:
!pip install python-dotenv

In [2]:
import os
from dotenv import load_dotenv
from pyspark.sql import SparkSession

load_dotenv()

True

In [3]:
usuario_minio = os.getenv("MINIO_ACCESS_KEY")
senha_minio = os.getenv("MINIO_SECRET_KEY")

In [4]:
spark = (
    SparkSession.builder
        .appName("teste-minio")
        .master("local[*]")
        #.master("spark://spark-master:7077")
        .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0,org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262")

        #config s3a --> minio
        .config("spark.hadoop.fs.s3a.endpoint", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
        .config("spark.hadoop.fs.s3a.access.key",usuario_minio)
        .config("spark.hadoop.fs.s3a.secret.key", senha_minio)
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
        .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")

        # Delta lake - obrigatório para usar format("delta")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .getOrCreate()
)

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ad547bd8-437c-4929-97eb-7bf3bc1b9056;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 256ms :: artifacts dl 21ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	io.delta#delta-spark_2.12;3.1.0 from central in [default]
	io.delta#delta-storage;3.1.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 

In [5]:
df = spark.read.csv("s3a://dados/E-Commerce/Categoria.csv", header=True, inferSchema=True)

26/07/14 17:49:59 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [6]:
df.show()

+---+--------------------+
| id|                name|
+---+--------------------+
|  0|   Moda e Acessórios|
|  1|Cosméticos e Perf...|
|  2|    Eletrodomésticos|
|  3|              Livros|
|  4|           Celulares|
|  5|         Informática|
|  6|    Casa e Decoração|
|  7|         Eletrônicos|
|  8|     Esporte e Lazer|
|  9|  Brinquedos e Games|
+---+--------------------+



In [7]:
(df.write
    .format("delta")
    .mode("overwrite")
    .save("s3a://dados/bronze/Categoria.delta")

)

26/07/14 17:50:10 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [9]:
spark.sql("""
    DESCRIBE DETAIL
    delta.`s3a://dados/bronze/Categoria.delta`
""").show(truncate=False)

+------+------------------------------------+----+-----------+----------------------------------+-----------------------+-------------------+----------------+--------+-----------+----------+----------------+----------------+------------------------+
|format|id                                  |name|description|location                          |createdAt              |lastModified       |partitionColumns|numFiles|sizeInBytes|properties|minReaderVersion|minWriterVersion|tableFeatures           |
+------+------------------------------------+----+-----------+----------------------------------+-----------------------+-------------------+----------------+--------+-----------+----------+----------------+----------------+------------------------+
|delta |f4d76a51-7f06-4a12-a821-695c32bfdde4|NULL|NULL       |s3a://dados/bronze/Categoria.delta|2026-07-14 17:37:37.364|2026-07-14 17:50:14|[]              |1       |958        |{}        |1               |2               |[appendOnly, invariants]|


In [10]:
spark.sql("""
    DESCRIBE extended
    delta.`s3a://dados/bronze/Categoria.delta`
""")

DataFrame[col_name: string, data_type: string, comment: string]

In [11]:
spark.sql("""
    DESCRIBE history
    delta.`s3a://dados/bronze/Categoria.delta`
""").show(truncate=False)

+-------+-------------------+------+--------+---------+--------------------------------------+----+--------+---------+-----------+--------------+-------------+-----------------------------------------------------------+------------+-----------------------------------+
|version|timestamp          |userId|userName|operation|operationParameters                   |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                                           |userMetadata|engineInfo                         |
+-------+-------------------+------+--------+---------+--------------------------------------+----+--------+---------+-----------+--------------+-------------+-----------------------------------------------------------+------------+-----------------------------------+
|1      |2026-07-14 17:50:14|NULL  |NULL    |WRITE    |{mode -> Overwrite, partitionBy -> []}|NULL|NULL    |NULL     |0          |Serializable  |false        |{numFiles -> 1, numOutputRows -> 1

In [18]:
spark.stop()